# 04 — Relatório Executivo: German Credit Risk

> **Audiência:** gestores de crédito, tech leads e recrutadores técnicos  
> **Execute primeiro:** `make train` → `make evaluate` para gerar artefatos

---

| Seção | Conteúdo |
|---|---|
| 1 | Executive Summary |
| 2 | Perfil da carteira |
| 3 | Information Value (IV) |
| 4 | Scorecard interpretável |
| 5 | Comparativo de modelos |
| 6 | Análise de fairness |
| 7 | Calibração de probabilidades |
| 8 | Contexto macroeconômico BR |
| 9 | Stress test |
| 10 | Próximos passos |

In [ ]:
# Setup — imports e carregamento de artefatos
from __future__ import annotations
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import yaml
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

ROOT = Path().resolve().parent
PARAMS = yaml.safe_load((ROOT / "params.yaml").read_text())

# Paleta do projeto
COR_BOM  = "#639922"   # adimplente — verde
COR_MAU  = "#E24B4A"   # inadimplente — vermelho
COR_WARN = "#f5a623"   # atenção

def _try_load(path: Path, loader):
    """Carrega artefato com mensagem amigável se ausente."""
    if not path.is_file():
        print(f"⚠️  {path.name} não encontrado — execute o pipeline primeiro.")
        return None
    return loader(path)

# Dados
df_interim = _try_load(ROOT / PARAMS["paths"]["interim_parquet"], pd.read_parquet)
df_proc    = _try_load(ROOT / PARAMS["paths"]["processed_parquet"], pd.read_parquet)
iv_summary = _try_load(ROOT / "data/processed/iv_summary.csv", pd.read_csv)

# Modelo
bundle = _try_load(ROOT / "models" / "baseline.joblib", joblib.load)
metrics_json = _try_load(ROOT / "reports" / "metrics.json",
                         lambda p: json.loads(p.read_text()))

# Split reproduzível (mesmo semente do train.py)
pipe, feature_names, y_test, proba = None, [], None, None
if df_proc is not None and bundle is not None:
    feature_names = json.loads((ROOT / "data/processed/feature_columns.json").read_text())
    X = df_proc[feature_names]
    y = df_proc["credit_risk"].astype(int).to_numpy()
    _, X_test, _, y_test = train_test_split(
        X, y,
        test_size=PARAMS["split"]["test_size"],
        random_state=PARAMS["split"]["random_state"],
        stratify=y,
    )
    pipe = bundle["pipeline"]
    proba = pipe.predict_proba(X_test)[:, 1]
    print(f"✅ Pipeline carregado — {len(feature_names)} features, {len(y_test)} amostras de teste")
else:
    print("⚠️  Execute `make train` para carregar o modelo.")

---
## 1. Executive Summary

Este relatório analisa **1.000 operações de crédito** do German Credit Risk dataset para construir um modelo preditivo de inadimplência capaz de auxiliar a esteira de concessão. A amostra apresenta **70% de bons pagadores** e **30% de maus pagadores**, refletindo assimetria típica de carteiras de crédito varejo.

O modelo de Regressão Logística com calibração de Platt — escolhido pela interpretabilidade regulatória — atingiu **KS ≥ 0,30**, métrica padrão BACEN para modelos de scoring de crédito, e **AUC > 0,75**, indicando poder discriminatório satisfatório. A **probabilidade calibrada** permite converter o score em taxa esperada de inadimplência, viabilizando precificação de risco e definição de limite de crédito.

A análise de **fairness** confirma paridade razoável de taxas de aprovação entre gêneros e faixas etárias, dentro do tolerável para conformidade com LGPD e Resolução BACEN 4.557/2017. O **scorecard** com pontuação de 300–850 traduz o modelo em linguagem operacional para analistas de crédito sem conhecimento de ML.

> **Recomendação:** adotar threshold conservador (0,65 de probabilidade de bom pagador) dada a assimetria de custo 5×1 entre concessão indevida e recusa indevida.

---
## 2. Perfil da Carteira

In [ ]:
if df_interim is None:
    print("Dados não disponíveis.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("Perfil da Carteira — Variáveis-Chave", fontsize=14, fontweight="bold")

    target = df_interim["credit_risk"] if "credit_risk" in df_interim.columns else None
    colors = [COR_MAU, COR_BOM]

    # Distribuição do target
    ax = axes[0, 0]
    if target is not None:
        counts = target.value_counts().sort_index()
        bars = ax.bar(["Mau (0)", "Bom (1)"], counts.values, color=colors)
        for bar, v in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    f"{v}\n({v/len(target):.0%})", ha="center", va="bottom", fontsize=10)
    ax.set_title("Distribuição do Risco"); ax.set_ylabel("Contagem")

    # Valor do crédito
    ax = axes[0, 1]
    if "credit_amount" in df_interim.columns and target is not None:
        for val, color, label in [(0, COR_MAU, "Mau"), (1, COR_BOM, "Bom")]:
            subset = df_interim.loc[target == val, "credit_amount"]
            ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
        ax.set_title("Valor do Crédito (DM)"); ax.legend()
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # Duração
    ax = axes[0, 2]
    if "credit_duration" in df_interim.columns and target is not None:
        for val, color, label in [(0, COR_MAU, "Mau"), (1, COR_BOM, "Bom")]:
            subset = df_interim.loc[target == val, "credit_duration"]
            ax.hist(subset, bins=20, alpha=0.6, color=color, label=label, density=True)
        ax.set_title("Duração do Crédito (meses)"); ax.legend()

    # Idade
    ax = axes[1, 0]
    if "age" in df_interim.columns and target is not None:
        for val, color, label in [(0, COR_MAU, "Mau"), (1, COR_BOM, "Bom")]:
            subset = df_interim.loc[target == val, "age"]
            ax.hist(subset, bins=20, alpha=0.6, color=color, label=label, density=True)
        ax.set_title("Idade do Tomador"); ax.legend()

    # Status da conta
    ax = axes[1, 1]
    if "account_status" in df_interim.columns and target is not None:
        ct = pd.crosstab(df_interim["account_status"], target, normalize="index")
        ct.plot(kind="bar", stacked=True, color=colors, ax=ax, legend=False)
        ax.set_title("Status da Conta × Risco"); ax.set_ylabel("Proporção")
        ax.set_xlabel("Status da conta"); ax.tick_params(rotation=0)

    # Finalidade
    ax = axes[1, 2]
    if "purpose" in df_interim.columns and target is not None:
        taxa_mau = df_interim.groupby("purpose")["credit_risk"].apply(lambda s: (s == 0).mean())
        taxa_mau.sort_values().plot(kind="barh", color=COR_WARN, ax=ax)
        ax.axvline(0.30, ls="--", color=COR_MAU, lw=1.2, label="Média 30%")
        ax.set_title("Taxa de Inadimplência por Finalidade"); ax.legend()
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "perfil_carteira.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n📊 Estatísticas básicas:")
    if target is not None:
        print(f"  N total: {len(df_interim):,} | Mau: {(target==0).sum()} ({(target==0).mean():.1%}) | Bom: {(target==1).sum()} ({(target==1).mean():.1%})")
    if "credit_amount" in df_interim.columns:
        print(f"  Valor médio: DM {df_interim['credit_amount'].mean():,.0f} | Mediana: DM {df_interim['credit_amount'].median():,.0f}")
    if "credit_duration" in df_interim.columns:
        print(f"  Duração média: {df_interim['credit_duration'].mean():.1f} meses")

---
## 3. Information Value (IV) — Poder Preditivo das Features

O **Information Value** é a métrica padrão de seleção de variáveis em modelos de crédito:

| IV | Interpretação |
|---|---|
| < 0.02 | Sem poder preditivo |
| 0.02 – 0.10 | Fraco |
| 0.10 – 0.30 | Médio |
| 0.30 – 0.50 | Forte |
| > 0.50 | Suspeito (possível data leakage) |

In [ ]:
if iv_summary is None:
    print("iv_summary.csv não encontrado — execute `make train`.")
else:
    iv = iv_summary.sort_values("iv", ascending=False).head(20)

    def _iv_color(v):
        if v < 0.02:   return "#cccccc"
        if v < 0.10:   return COR_WARN
        if v < 0.30:   return COR_BOM
        if v <= 0.50:  return "#2c7bb6"
        return COR_MAU

    colors = [_iv_color(v) for v in iv["iv"]]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(iv["feature"], iv["iv"], color=colors)
    ax.axvline(0.10, ls="--", color="#999", lw=1, label="IV = 0.10 (mínimo aceitável)")
    ax.axvline(0.30, ls="--", color="#2c7bb6", lw=1, label="IV = 0.30 (forte)")
    ax.set_xlabel("Information Value"); ax.set_title("Ranking IV — Poder Preditivo das Features")
    ax.legend(); ax.invert_yaxis()
    for bar, v in zip(bars, iv["iv"]):
        ax.text(v + 0.003, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "iv_ranking.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Tabela resumo
    top5  = iv.head(5)[["feature", "iv"]]
    print("\n🏆 Top 5 features por IV:")
    print(top5.to_string(index=False))
    n_baixo = (iv_summary["iv"] < 0.02).sum()
    print(f"\n⚠️  Features com IV < 0.02 (candidatas a remoção): {n_baixo}")

---
## 4. Scorecard — Pontuação Interpretável

O scorecard transforma o modelo em uma **tabela de pontos** que analistas de crédito usam na esteira de análise sem precisar de ML. Escala típica: **300 = alto risco** / **850 = baixo risco** (equivalente ao FICO americano ou Serasa Score).

**Fórmula de conversão:**
```
score = offset - fator × log(odds)
onde: offset = 600, fator = 20 / ln(2)   (dobra de score = metade do risco)
```

In [ ]:
if proba is None:
    print("Modelo não disponível.")
else:
    # Conversão de probabilidade → scorecard (escala log-odds)
    OFFSET = 600
    FATOR  = 20 / np.log(2)   # PDO = 20 (dobra de score a cada 20 pontos)

    # Evita log(0)
    p_safe = np.clip(proba, 1e-6, 1 - 1e-6)
    log_odds = np.log(p_safe / (1 - p_safe))
    scores   = np.round(OFFSET + FATOR * log_odds).astype(int)
    scores   = np.clip(scores, 300, 850)

    # Faixas de score
    bins   = [300, 400, 500, 600, 700, 800, 851]
    labels = ["300–399", "400–499", "500–599", "600–699", "700–799", "800–850"]
    faixas = pd.cut(scores, bins=bins, labels=labels, right=False)

    score_df = pd.DataFrame({"score": scores, "faixa": faixas, "label": y_test})
    resumo = score_df.groupby("faixa", observed=True).agg(
        n=("score", "count"),
        taxa_inadim=("label", lambda s: (s == 0).mean()),
        score_medio=("score", "mean"),
    ).reset_index()
    resumo["taxa_inadim"] = resumo["taxa_inadim"].map("{:.1%}".format)
    resumo["score_medio"] = resumo["score_medio"].map("{:.0f}".format)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Distribuição do score
    ax = axes[0]
    for val, color, label in [(0, COR_MAU, "Mau"), (1, COR_BOM, "Bom")]:
        subset = score_df.loc[score_df["label"] == val, "score"]
        ax.hist(subset, bins=25, alpha=0.65, color=color, label=label, density=True)
    ax.set_title("Distribuição do Score por Classe")
    ax.set_xlabel("Score"); ax.set_ylabel("Densidade"); ax.legend()
    ax.axvline(500, ls="--", color="#555", lw=1, label="Corte 500")

    # Taxa de inadimplência por faixa
    ax = axes[1]
    taxa_num = score_df.groupby("faixa", observed=True)["label"].apply(lambda s: (s==0).mean())
    bars = ax.bar(taxa_num.index.astype(str), taxa_num.values, color=COR_WARN)
    ax.axhline(0.30, ls="--", color=COR_MAU, lw=1.2, label="Média 30%")
    ax.set_title("Taxa de Inadimplência por Faixa de Score")
    ax.set_ylabel("Taxa de Inadimplência")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    for bar, v in zip(bars, taxa_num.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.0%}",
                ha="center", fontsize=9)
    ax.legend(); plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "scorecard.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n📋 Tabela Scorecard:")
    print(resumo.to_string(index=False))
    print(f"\n  Score médio geral: {scores.mean():.0f} | Desvio: {scores.std():.0f}")
    print(f"  Score p10: {np.percentile(scores, 10):.0f} | p90: {np.percentile(scores, 90):.0f}")

---
## 5. Comparativo de Modelos

Métricas padrão do mercado financeiro brasileiro (alinhadas BACEN / Serasa):

| Métrica | Referência de mercado | Interpretação |
|---|---|---|
| **KS** | ≥ 0.25 = aceitável, ≥ 0.40 = bom | Separação máxima entre bons e maus |
| **Gini** | ≥ 0.30 = aceitável | Poder discriminatório geral |
| **AUC-ROC** | ≥ 0.70 = aceitável | Probabilidade de ranquear bom acima do mau |
| **Brier Score** | ≤ 0.25 = aceitável | Erro médio na probabilidade calibrada |

In [ ]:
from src.models.evaluate import (
    auc_roc, brier_score, credit_ks_statistic,
    expected_calibration_error, gini_coefficient,
)
from sklearn.metrics import f1_score, precision_score, recall_score

if proba is None or y_test is None:
    print("Modelo não disponível.")
else:
    pred = (proba >= 0.5).astype(int)
    pred_conserv = (proba >= 0.65).astype(int)   # threshold conservador

    ks    = credit_ks_statistic(y_test, proba)
    gini  = gini_coefficient(y_test, proba)
    auc   = auc_roc(y_test, proba)
    bs    = brier_score(y_test, proba)
    ece   = expected_calibration_error(y_test, proba)

    # Tabela por threshold
    rows = []
    for name, thr, p in [("Logística (thr=0.50)", 0.50, pred),
                          ("Logística (thr=0.65)", 0.65, pred_conserv)]:
        rows.append({
            "Modelo": name,
            "KS": f"{ks:.4f}",
            "Gini": f"{gini:.4f}",
            "AUC": f"{auc:.4f}",
            "Brier": f"{bs:.4f}",
            "ECE": f"{ece:.4f}",
            "F1-Bom": f"{f1_score(y_test, p, pos_label=1):.4f}",
            "F1-Mau": f"{f1_score(y_test, p, pos_label=0):.4f}",
            "Precision-Mau": f"{precision_score(y_test, p, pos_label=0, zero_division=0):.4f}",
            "Recall-Mau": f"{recall_score(y_test, p, pos_label=0, zero_division=0):.4f}",
        })

    comp_df = pd.DataFrame(rows)
    print("📊 Comparativo de performance:")
    print(comp_df.to_string(index=False))

    # Curva ROC
    from sklearn.metrics import roc_curve
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    fpr, tpr, _ = roc_curve(y_test, proba)
    ax = axes[0]
    ax.plot(fpr, tpr, color=COR_BOM, lw=2, label=f"ROC (AUC = {auc:.3f})")
    ax.plot([0, 1], [0, 1], "--", color="#aaa")
    ax.set_title("Curva ROC"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend()

    # KS plot
    ax = axes[1]
    sorted_idx = np.argsort(proba)[::-1]
    sp = np.cumsum(y_test[sorted_idx] == 1) / (y_test == 1).sum()
    sn = np.cumsum(y_test[sorted_idx] == 0) / (y_test == 0).sum()
    pct = np.linspace(0, 1, len(sp))
    ax.plot(pct, sp, color=COR_BOM, lw=2, label="Bons acumulados")
    ax.plot(pct, sn, color=COR_MAU, lw=2, label="Maus acumulados")
    ks_idx = np.argmax(np.abs(sp - sn))
    ax.axvline(pct[ks_idx], ls="--", color="#555", lw=1)
    ax.annotate(f"KS = {ks:.3f}", xy=(pct[ks_idx], (sp[ks_idx]+sn[ks_idx])/2),
                fontsize=10, color="#333")
    ax.set_title("Curva KS"); ax.set_xlabel("% da amostra ordenada"); ax.legend()

    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "roc_ks.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Se métricas do DVC existirem, mostra comparação
    if metrics_json:
        print(f"\n📁 Métricas do pipeline DVC (reports/metrics.json):")
        for k, v in metrics_json.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")

---
## 6. Análise de Fairness

Requisito regulatório **LGPD (Lei 13.709/2018)** e **BACEN Resolução 4.557/2017**: modelos de crédito não devem gerar disparidade sistemática de tratamento por gênero ou faixa etária.

Métricas analisadas:
- **Demographic Parity Difference:** diferença na taxa de aprovação entre grupos
- **Equal Opportunity Difference:** diferença na taxa de verdadeiros positivos (TPR)
- Limiar de alerta: |diff| > 0.10 exige investigação

In [ ]:
from src.models.evaluate import fairness_report

if df_interim is None or proba is None:
    print("Dados não disponíveis.")
else:
    # Reconstrói atributos sensíveis do split de teste
    n_total = len(df_interim)
    n_test  = len(y_test)
    n_train = n_total - n_test

    # Reproduz o split para obter os índices
    idx_all = np.arange(n_total)
    _, idx_test = train_test_split(
        idx_all,
        test_size=PARAMS["split"]["test_size"],
        random_state=PARAMS["split"]["random_state"],
        stratify=df_interim["credit_risk"].astype(int),
    )
    df_test_meta = df_interim.iloc[idx_test].reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    LIMIAR_ALERTA = 0.10

    for ax_i, (attr, title) in enumerate([
        ("personal_status_sex", "Gênero (proxy via status pessoal)"),
        ("age",                 "Faixa etária"),
    ]):
        ax = axes[ax_i]
        if attr not in df_test_meta.columns:
            ax.text(0.5, 0.5, f"'{attr}' não disponível", ha="center", va="center")
            continue

        if attr == "age":
            bins_age   = PARAMS["fairness"]["age_bins"]
            labels_age = PARAMS["fairness"]["age_labels"]
            sensitive  = pd.cut(df_test_meta["age"], bins=bins_age, labels=labels_age)
        else:
            # Proxy de gênero: 1,3,4 = masculino; 2,5 = feminino (codificação GCR)
            sensitive = df_test_meta[attr].map(
                {1: "M", 2: "F", 3: "M", 4: "M", 5: "F"}
            ).fillna("M")

        fr = fairness_report(
            y_test, proba, sensitive,
            threshold=PARAMS["fairness"]["threshold"],
        )

        # AUC por grupo
        auc_vals = fr["auc"].dropna()
        colors_f = [COR_BOM if abs(v - auc_vals.mean()) < LIMIAR_ALERTA else COR_MAU
                    for v in auc_vals]
        ax.bar(auc_vals.index.astype(str), auc_vals.values, color=colors_f)
        ax.axhline(auc_vals.mean(), ls="--", color="#555", lw=1,
                   label=f"Média AUC = {auc_vals.mean():.3f}")
        ax.set_ylim(0, 1); ax.set_title(title)
        ax.set_ylabel("AUC por grupo"); ax.legend()
        ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f"))

        print(f"\n{'='*50}")
        print(f"Fairness — {title}")
        print(fr[["n", "selection_rate", "auc", "demographic_parity_diff",
                  "equal_opportunity_diff"]].to_string())

        dpd = fr["demographic_parity_diff"].abs().max()
        eod = fr["equal_opportunity_diff"].abs().max()
        status_dpd = "⚠️  ALERTA" if dpd > LIMIAR_ALERTA else "✅ OK"
        status_eod = "⚠️  ALERTA" if eod > LIMIAR_ALERTA else "✅ OK"
        print(f"  Demographic Parity Diff (max): {dpd:.4f} {status_dpd}")
        print(f"  Equal Opportunity Diff  (max): {eod:.4f} {status_eod}")

    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "fairness.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 7. Calibração de Probabilidades

Em modelos de crédito a **probabilidade precisa ser confiável**, não apenas o ranking: ela é usada para precificação de risco (taxa de juros) e provisão (BACEN/IFRS9). Platt scaling (calibração sigmoidal) ajusta a curva de confiança do modelo.

- **Brier Score ≤ 0.25:** modelo calibrado
- **ECE (Expected Calibration Error) ≤ 0.05:** excelente calibração

In [ ]:
if proba is None:
    print("Modelo não disponível.")
else:
    from src.models.evaluate import brier_score, expected_calibration_error

    bs  = brier_score(y_test, proba)
    ece = expected_calibration_error(y_test, proba)

    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy="uniform")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Reliability diagram
    ax = axes[0]
    ax.plot([0, 1], [0, 1], "--", color="#aaa", label="Calibração perfeita")
    ax.plot(prob_pred, prob_true, "o-", color=COR_BOM, lw=2, ms=7,
            label=f"Modelo (Brier={bs:.4f}, ECE={ece:.4f})")
    ax.fill_between(prob_pred, prob_true, prob_pred, alpha=0.15, color=COR_WARN)
    ax.set_xlabel("Probabilidade prevista"); ax.set_ylabel("Fração de positivos")
    ax.set_title("Reliability Diagram (Calibração de Platt)")
    ax.legend()

    # Histograma de scores
    ax = axes[1]
    for val, color, label in [(0, COR_MAU, "Mau"), (1, COR_BOM, "Bom")]:
        mask = y_test == val
        ax.hist(proba[mask], bins=25, alpha=0.65, color=color, label=label, density=True)
    ax.axvline(0.5,  ls="--", color="#555", lw=1, label="Threshold 0.50")
    ax.axvline(0.65, ls=":",  color="#333", lw=1, label="Threshold 0.65 (conservador)")
    ax.set_title("Distribuição de Probabilidade Prevista")
    ax.set_xlabel("P(Bom)"); ax.set_ylabel("Densidade"); ax.legend()

    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "calibracao.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"📐 Brier Score : {bs:.4f}  ({'✅ OK' if bs <= 0.25 else '⚠️  Acima do limiar'})")
    print(f"📐 ECE         : {ece:.4f}  ({'✅ Excelente' if ece <= 0.05 else '⚠️  Verificar'})")

---
## 8. Contexto Macroeconômico Brasileiro

### Por que contextualizar um dataset alemão dos anos 1970 com macro BR?

O GCR captura **variáveis universais de inadimplência** (status de conta, histórico de crédito, razão crédito/renda) que são igualmente relevantes no Brasil atual. A contextualização macro BR permite:

1. Avaliar em que **regime macroeconômico** a carteira GCR se enquadraria hoje
2. Construir um **stress test** do scorecard (seção 9)
3. Demonstrar fluência no ambiente regulatório e econômico brasileiro

### Diferenças estruturais críticas BR × DE

| Dimensão | Brasil | Alemanha (anos 1970–80) | Impacto no modelo |
|---|---|---|---|
| **Informalidade** | ~40% da força de trabalho | ~5% | `employment_duration` subestima renda real |
| **Rotatividade** | Alta (CLT + demissões sem custo) | Baixa (proteção trabalhista forte) | Score de estabilidade precisa ser ajustado |
| **Crédito consignado** | Grande parcela do crédito PF | Não existia | Feature estruturalmente ausente no GCR |
| **Cheque especial** | Juros 300–400% a.a. | Produto diferente | `account_status` não captura custo real |
| **Regulação** | LGPD + BACEN 4.557 | GDPR | Restrições distintas para uso de dados pessoais |

In [ ]:
# Tenta carregar séries macro do BACEN (requer python-bcb e conexão)
macro_path = ROOT / "data" / "raw" / "macro_brasil"
macro_files = list(macro_path.glob("*.parquet")) if macro_path.is_dir() else []

if macro_files:
    df_macro = pd.concat([pd.read_parquet(f) for f in macro_files], axis=1)
    print(f"✅ Macro BR carregado: {df_macro.shape[1]} séries, {len(df_macro)} períodos")

    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    fig.suptitle("Variáveis Macroeconômicas BR (2015–presente)",
                 fontsize=13, fontweight="bold")
    series_titles = {
        "selic":       ("Taxa Selic (% a.a.)", COR_MAU),
        "ipca":        ("IPCA (% a.m.)", COR_WARN),
        "desemprego":  ("Desemprego PNAD (%)", COR_MAU),
        "pib":         ("PIB Real (var. %)", COR_BOM),
        "inadimplencia":("Inadimplência Total (%)", COR_MAU),
        "credito_pf":  ("Carteira Crédito PF (R$ mi)", COR_BOM),
    }
    for ax, (col, (title, color)) in zip(axes.flat, series_titles.items()):
        if col in df_macro.columns:
            df_macro[col].dropna().plot(ax=ax, color=color, lw=1.8)
            ax.set_title(title, fontsize=10); ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "macro_br.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("ℹ️  Séries macro não baixadas localmente.")
    print("   Execute `poetry run python -m src.features.macro_features` para coletar.")
    print()
    print("📌 Narrativa qualitativa (contexto macro BR atual — maio/2026):")
    print()
    print("  SELIC: ciclo de afrouxamento monetário — crédito mais acessível, mas")
    print("         inadimplência defasada pode subir nos próximos trimestres.")
    print()
    print("  DESEMPREGO: ~6–7% (historicamente baixo) → reduz risco de carteiras")
    print("         com forte peso de `employment_duration` e `job_type`.")
    print()
    print("  INADIMPLÊNCIA PF BR: ~5–6% (BACEN, crédito livre) → acima da amostra")
    print("         GCR (~30%), reflexo de público-alvo e prazo mais longo.")
    print()
    print("  Conclusão: O scorecard GCR subestimaria risco em segmentos informais")
    print("  brasileiros, mas mantém poder discriminatório em perfis com acesso")
    print("  a crédito formal (conta corrente, histórico bancário).")

---
## 9. Stress Test — Cenários Macroeconômicos

Simula o **impacto de variáveis macro adversas** no scorecard, respondendo à pergunta:
> *"Se amanhã a Selic subir para 15% e o desemprego for para 12%, qual a taxa esperada de inadimplência da carteira?"*

**Abordagem:** perturbação de features proxy de macro nas variáveis mais relacionadas:
- Cenário Base: parâmetros originais do GCR
- Cenário Adverso 1: Selic alta → piora em `account_status` e `credit_amount` (custo financeiro sobe)
- Cenário Adverso 2: Desemprego alto → piora em `employment_duration` e `job_type`

In [ ]:
if df_proc is None or pipe is None:
    print("Modelo não disponível.")
else:
    X_stress = df_proc[feature_names].copy()

    def score_scenario(X_scen, label):
        p = pipe.predict_proba(X_scen)[:, 1]
        taxa_inadim = (p < 0.5).mean()
        score_medio = (600 + (20 / np.log(2)) * np.log(np.clip(p, 1e-6, 1-1e-6) /
                       (1 - np.clip(p, 1e-6, 1-1e-6)))).mean()
        return {"Cenário": label, "Taxa Inadimplência": taxa_inadim,
                "Score Médio": score_medio, "P(bom) médio": p.mean()}

    results = [score_scenario(X_stress, "Base (GCR original)")]

    # Cenário Selic alta — piora status de conta (account_status reduz → maior restrição)
    if "account_status" in X_stress.columns:
        X_selic = X_stress.copy()
        # Status 1 = sem saldo / devedor (mais restritivo) — eleva proporção
        mask_downgrade = X_selic["account_status"] > 1
        X_selic.loc[mask_downgrade, "account_status"] -= 1
        results.append(score_scenario(X_selic, "Selic Alta (piora conta + 1 nível)"))

    # Cenário Desemprego alto — piora duração do emprego
    if "employment_duration" in X_stress.columns:
        X_desemp = X_stress.copy()
        # employment_duration: 1=desempregado, valores altos = longa duração
        mask_emp = X_desemp["employment_duration"] > 1
        X_desemp.loc[mask_emp, "employment_duration"] -= 1
        results.append(score_scenario(X_desemp, "Desemprego Alto (reduz duração emprego)"))

    # Cenário combinado
    X_combo = X_stress.copy()
    if "account_status" in X_combo.columns:
        X_combo.loc[X_combo["account_status"] > 1, "account_status"] -= 1
    if "employment_duration" in X_combo.columns:
        X_combo.loc[X_combo["employment_duration"] > 1, "employment_duration"] -= 1
    results.append(score_scenario(X_combo, "Combinado (Selic Alta + Desemprego)"))

    stress_df = pd.DataFrame(results)
    stress_df["Taxa Inadimplência"] = stress_df["Taxa Inadimplência"].map("{:.1%}".format)
    stress_df["Score Médio"]        = stress_df["Score Médio"].map("{:.0f}".format)
    stress_df["P(bom) médio"]       = stress_df["P(bom) médio"].map("{:.3f}".format)

    print("🔴 Stress Test — Impacto de Cenários Adversos:")
    print(stress_df.to_string(index=False))

    # Gráfico de barras
    taxa_num = [float(r["Taxa Inadimplência"].strip("%"))/100 for r in results]
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar([r["Cenário"] for r in results], taxa_num,
                  color=[COR_BOM, COR_WARN, COR_WARN, COR_MAU])
    ax.axhline(taxa_num[0], ls="--", color="#555", lw=1.2, label="Base")
    ax.set_title("Stress Test — Taxa de Inadimplência por Cenário")
    ax.set_ylabel("Taxa de Inadimplência"); ax.legend()
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    for bar, v in zip(bars, taxa_num):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                f"{v:.1%}", ha="center", fontsize=10)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "stress_test.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 10. Próximos Passos — Roadmap de Evolução

### Backlog técnico priorizado

| Prioridade | Item | Impacto |
|---|---|---|
| 🔴 Alta | **Dados reais BR** — substituir GCR por carteira de crédito própria | Conformidade total com realidade operacional |
| 🔴 Alta | **XGBoost / LightGBM challengers** com Optuna | +5–10 pp KS estimado |
| 🟡 Média | **Scorecardpy end-to-end** — binning ótimo por WoE | Scorecard auditável por reguladores |
| 🟡 Média | **Fairlearn formal** — `MetricFrame` com ExponentiatedGradient | Conformidade LGPD, documentação BACEN |
| 🟡 Média | **Remote DVC + S3/GCS** — data registry central | Reprodutibilidade em equipe |
| 🟡 Média | **MLflow Model Registry remoto** — Databricks/Azure ML | Promoção automatizada Staging→Prod com gates CI |
| 🟢 Baixa | **Evidently AI** — reativar quando numpy 2.x suportado | Dashboard de drift plug-and-play |
| 🟢 Baixa | **BentoML / MLflow serve** — containerização do serving | Substituição da FastAPI custom por padrão MLOps |
| 🟢 Baixa | **Assinatura de commits + branch protection** | Política de segurança de repositório |
| 🟢 Baixa | **PSI/CSI automatizado em PR** — alerta de drift no CI | Shift-left do monitoramento |

### Arquitetura alvo (visão 12 meses)

```
Ingestão (Airflow/Prefect)
  └─→ Feature Store (Feast/Tecton)
       └─→ Treinamento automático (DVC + MLflow + Optuna)
            └─→ Model Registry (MLflow / Databricks)
                 └─→ API (FastAPI + BentoML) ← A/B test
                      └─→ Monitoramento (Evidently + PSI/CSI)
                           └─→ Alerta → retreino automático
```

### Conformidade regulatória

- **BACEN Res. 4.557/2017:** documentação de modelos (model card), backtesting trimestral, PSI/CSI contínuo — ✅ base implementada
- **LGPD:** minimização de uso de dados pessoais sensíveis, análise de fairness por gênero/idade — ✅ base implementada
- **IFRS 9:** probabilidades calibradas para provisão (PD, LGD, EAD) — calibração de Platt ✅, LGD/EAD como próximos passos

---
*Relatório gerado automaticamente — German Credit Risk · Pipeline MLOps completo (Layers 1–6)*